# P1: DESMUNTANT LA MENTIDA VISUAL

## PREPOCESSAMENT DE DADES

In [2]:
%pip install pandas pycountry plotly "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\usuari\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import pycountry
import plotly.express as px
import plotly.graph_objects as go

In [4]:
pd.set_option("display.max_colwidth", None)

In [5]:
df = pd.read_csv("dades/dades_grafic.csv")
df.head()

,Indicator Name,Indicator Code,Country Name,Country Code,Year,Value,Disaggregation
0,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2025,63.674,"female, Modeled, 15+"
1,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2024,63.643,"female, Modeled, 15+"
2,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2023,63.593,"female, Modeled, 15+"
3,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2022,59.831,"female, Modeled, 15+"
4,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2021,59.454,"female, Modeled, 15+"


In [6]:
df.columns

Index(['Indicator Name', 'Indicator Code', 'Country Name', 'Country Code',
       'Year', 'Value', 'Disaggregation'],
      dtype='object')

El format del csv original era peculiar (ja que hi ha diferents tipus de Indicator Name), per tant, primer faig una adaptació. Se selecciona només les dades del 2023 i es guarda per cada país el % de dones i el % d'homes. 
Per tant, el primer que faig és començar a processar el dataframe però treballant només amb les dades del 2023, ja que és l'any en que es basa el gràfic.

In [7]:
df_2023 = df[df["Year"] == 2023]
df_2023.shape

(2841, 7)

El primer que volia comprovar era si el nombre de files era equiparable al nombre de països. Mirant per sobre el csv em vaig adonar que hi havia diferents entrades segons si es tractava del "Labor force participation rate" male i female. Per tant, el nombre de països hauria de ser la meitat del nombre de files. 


In [8]:
rows = len(df_2023)
total_indicator_names = len(df_2023["Indicator Name"].unique())
print(f"Hi ha {total_indicator_names} Indicator Names diferents")
total_countries = len(df_2023["Country Name"].unique())
print(f"Hi ha {total_countries} països i {rows} files")


Hi ha 15 Indicator Names diferents
Hi ha 236 països i 2841 files


Tal i com s'observa, hi ha una gran diferència respecte el nombre d'entrades i el nombre de països del dataset. 
A partir d'aquí, vaig decidir agrupar pels diferents "Indicator Name", per veure si en cadascun hi havia aquests 262 països

També, comprovo que hi hagi el mateix nombre de Country Names que de Country Codes. D'aquesta manera, comprovo que no hi hagi més d'un codi associat a cada país de daltaframe .

In [9]:
if(len(df_2023["Country Name"].unique())==len(df_2023["Country Code"].unique())):
    print("Cada país té un codi associat.")
else:
    print("Hi ha discrepàncies entre el nombre de països i el nombre de codis")

Cada país té un codi associat.


Comprovo que realment un mateix country code només està associat a un país

In [10]:

paisos_codis = df.groupby("Country Name")["Country Code"].nunique()
if(max(paisos_codis)>1):
    print("Hi ha països associats a més d'un codi")
else:
    print("Cada país té un únic codi associat.")

Cada país té un únic codi associat.


In [11]:
indicador_names = df_2023.groupby("Indicator Name")
indicador_names.size()

Indicator Name
Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)                     231
Labor force participation rate for ages 15-24, female (%) (national estimate)                        127
Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)                       231
Labor force participation rate for ages 15-24, male (%) (national estimate)                          127
Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)                      231
Labor force participation rate for ages 15-24, total (%) (national estimate)                         127
Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)      231
Labor force participation rate, female (% of female population ages 15+) (national estimate)         127
Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)    231
Labor force participation rate, male (% 

En el gràfic posa escrit "Figures are rounded and based on the International Labour Organization's estimates as of 2023", per tant, treballaré amb aquells que indiquin "modeled ILO estimate". A partir d'aquí, cal averiguar quin és el rang d'edat (15-24, 15-64 o 15+) que es mostra al gràfic (ja que no està indicat).
Per comprovar quin rang d'edat s'utilitza en el gràfic, comparo les dades d'Afganistan per als diferents intervals disponibles. El gràfic mostra un 100% de participació laboral masculina i un 5% femenina, per tant, busco quin dels indicadors s'aproxima a aquests valors.

In [12]:
afghanistan=df_2023[df_2023["Country Name"] == "Afghanistan"]
afghanistan=afghanistan[afghanistan["Indicator Name"].str.contains("ILO estimate")]
afghanistan = afghanistan[["Indicator Name","Value"]]
afghanistan

,Indicator Name,Value
1694,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",5.145
15611,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",70.074
29527,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",37.588
43670,"Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)",6.013
56953,"Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)",68.598
70236,"Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)",38.006
83544,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567


Tal i com s'observa, en el % d'homes en cap de les tres opcions coincideix amb el 100% que es mostra el gràfic, ja que els valors se situen entre el 68,598% i el 71,384%.
En canvi, en el % de dones, sí que hi ha el 5% en les entrades on l'edat és de 15-64 o de 15+. 
Per tant, miro pels 12 paísos del gràfic tots els valors, per veure si és un cas aïllat o és per tots els països.

In [13]:
#Tots els noms dels països del gràfic i del CSV no coincideixen, però amb agafant els nous noms ja s'agafen
paisos = [
    "Afghanistan",
    "Iraq",
    "Pakistan",
    "Oman",
    "Iran, Islamic Rep.",
    "Yemen, Rep.",
    "Egypt, Arab Rep.",
    "Algeria",
    "Syrian Arab Republic",
    "Jordan",
    "Morocco",
    "Saudi Arabia"
]
df_12 = df_2023[df_2023["Country Name"].isin(paisos)]
df_12 = df_12[df_12["Indicator Name"].str.contains("modeled ILO estimate")]
df_12_15 = df_12[df_12["Indicator Name"].str.contains("population ages 15-64",regex=False)]
df_12_15 = df_12_15.sort_values(["Country Name", "Indicator Name"])
df_12_15[["Country Name", "Indicator Name", "Value"]]


,Country Name,Indicator Name,Value
83544,Afghanistan,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,Afghanistan,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,Afghanistan,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567
83616,Algeria,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",15.834
92026,Algeria,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",73.378
100436,Algeria,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",45.346
85308,"Egypt, Arab Rep.","Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",19.830
93718,"Egypt, Arab Rep.","Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",74.050
102128,"Egypt, Arab Rep.","Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",47.332
86352,"Iran, Islamic Rep.","Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",15.650


In [14]:
afghanistan=df_2023[df_2023["Country Name"] == "Afghanistan"]
afghanistan=afghanistan[afghanistan["Indicator Name"].str.contains("ILO estimate")]
afghanistan = afghanistan[["Indicator Name","Value"]]
afghanistan

,Indicator Name,Value
1694,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",5.145
15611,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",70.074
29527,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",37.588
43670,"Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)",6.013
56953,"Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)",68.598
70236,"Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)",38.006
83544,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567


Tal i com s'observa, alguns valors coincideixen, però no tots. He provat agafant els diferents tipus de rangs d'edats i al final he escollit el de 15-64.
Són amb aquests valors que contrueixo un nou dataframe transposat per poder després fer el gràfic. 

In [15]:
df_grafic = df_12_15.pivot(
    index="Country Name",
    columns="Indicator Name",
    values="Value"
)
df_grafic = df_grafic.rename(columns={
    "Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)": "Female",
    "Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)": "Male",
    "Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)": "Total"
})

df_grafic = df_grafic.reset_index()
df_grafic.columns.name = None
df_grafic

,Country Name,Female,Male,Total
0,Afghanistan,5.359,71.384,38.567
1,Algeria,15.834,73.378,45.346
2,"Egypt, Arab Rep.",19.830,74.050,47.332
3,"Iran, Islamic Rep.",15.650,73.892,45.302
4,Iraq,11.505,75.346,43.445
5,Jordan,17.057,66.084,42.620
6,Morocco,21.374,73.674,47.784
7,Oman,31.945,89.774,70.389
8,Pakistan,25.475,82.873,54.686
9,Saudi Arabia,35.617,84.185,66.724


Alguns dels valors són similars, però sí que per exemple en països com Afganistan, es passa d'un 100% a un 70%.
Com que això m'ha sorprès, he decidit fer una nova taula comparativa. En aquesta hi ha els valors del gràfic escrits manualment, per detectar i demostrar aquestes discrepàncies. 

In [16]:
grafic_original = pd.DataFrame({
    "Country Name": paisos,
    "Grafic_Female": [5, 11, 25, 32, 14, 5, 17, 17, 14, 14, 20, 35],
    "Grafic_Male": [100, 92, 81, 88, 71, 60, 71, 67, 64, 63, 68, 80]
})
comparacio = df_grafic.merge(grafic_original, on="Country Name")
#Comparo els valors del gràfic amb els del dataset per veure'n les diferències.
comparacio["Dif_Female"] = abs(comparacio["Grafic_Female"] - comparacio["Female"])
comparacio["Dif_Male"] = abs(comparacio["Grafic_Male"] - comparacio["Male"])
comparacio = comparacio[["Country Name", "Female", "Grafic_Female", "Dif_Female", "Male", "Grafic_Male","Dif_Male"]]
comparacio


,Country Name,Female,Grafic_Female,Dif_Female,Male,Grafic_Male,Dif_Male
0,Afghanistan,5.359,5,0.359,71.384,100,28.616
1,Algeria,15.834,17,1.166,73.378,67,6.378
2,"Egypt, Arab Rep.",19.830,17,2.830,74.050,71,3.050
3,"Iran, Islamic Rep.",15.650,14,1.650,73.892,71,2.892
4,Iraq,11.505,11,0.505,75.346,92,16.654
5,Jordan,17.057,14,3.057,66.084,63,3.084
6,Morocco,21.374,20,1.374,73.674,68,5.674
7,Oman,31.945,32,0.055,89.774,88,1.774
8,Pakistan,25.475,25,0.475,82.873,81,1.873
9,Saudi Arabia,35.617,35,0.617,84.185,80,4.185


Tal i com s'observa, alguns valors coincideixen aproximadament, però alguns altres presenten diferències importants. Per exemple, a Iraq, el gràfic mostra un 92% d'homes, mentre que en el CSV es donen valors de 75,346% per als homes i 11,505% per a les dones. Passa igual, com ja he mencionat, amb Afganistan, es passa d'un 100% del gràfic a un 71,384%.
Per tant, a partir d'ara treballaré amb el dataframe corregit, que correspon a: df_grafic

Considerant que els valors del gràfic no està clar d'on provenen, he volgut comprovar que els 12 països seleccionats siguin realment els 12 països amb  major diferència entre la participació laboral masculina i femenina segons les dades de l'ILO.
Analitzant els 12 països amb major diferència a nivell mundial, van aparèixer valors de regions i no de països, com per exemple Middle East & North Africa (IDA & IBRD), Middle East & North Africa (excluding high income), Arab World...
En python existeix una biblioteca per identificar països, per tant l'he aprofitat per seleccionar només aquelles files que corresponguessin a algun país segons el seu codi ISO (columna "Country Code" del dataframe)


In [17]:

#Creem una llista amb els codis ISO dels països
codis_paisos = {pais.alpha_3 for pais in pycountry.countries}
#Filtrem les dades del 2023 per a l'OIT i la població de 15-64 anys
df_mundial = df_2023[
    df_2023["Indicator Name"].str.contains("modeled ILO estimate", na=False)
    & df_2023["Indicator Name"].str.contains("population ages 15-64", regex=False, na=False)
].copy()
# Ens quedem només amb els països individuals
df_mundial = df_mundial[df_mundial["Country Code"].isin(codis_paisos)]

# Posem cada país en una fila i els indicadors en columnes
df_mundial = df_mundial.pivot(
    index=["Country Name", "Country Code"],
    columns="Indicator Name",
    values="Value"
).reset_index()

# Simplifiquem els noms de les columnes
df_mundial = df_mundial.rename(columns={
    "Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)": "Female",
    "Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)": "Male",
    "Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)": "Total"
})

# Traiem el nom de les columnes
df_mundial.columns.name = None

# Calculem la diferència entre homes i dones
df_mundial["Difference"] = df_mundial["Male"] - df_mundial["Female"]

# Ordenem els resultats i agafem els 12 primers
df_top12_mundial = df_mundial.sort_values(
    "Difference", ascending=False
).head(12)

# Mostrem les dades principals
df_top12_mundial[
    ["Country Name", "Country Code", "Female", "Male", "Difference"]
]

,Country Name,Country Code,Female,Male,Difference
0,Afghanistan,AFG,5.359,71.384,66.025
78,Iraq,IRQ,11.505,75.346,63.841
77,"Iran, Islamic Rep.",IRN,15.650,73.892,58.242
125,Oman,OMN,31.945,89.774,57.829
2,Algeria,DZA,15.834,73.378,57.544
180,"Yemen, Rep.",YEM,4.938,62.400,57.462
126,Pakistan,PAK,25.475,82.873,57.398
48,"Egypt, Arab Rep.",EGY,19.830,74.050,54.220
112,Morocco,MAR,21.374,73.674,52.300
159,Syrian Arab Republic,SYR,13.790,64.536,50.746


Tal i com s'observa, són els mateixos 12 països mostrats en el gràfic. 
El següent pas és mirar quina és la millor manera d'ordenar les dades de la gràfica. 
En el gràfic original estan ordenats tal que aixi (Afghanistan, Iraq, Pakistan, Oman, Iran, Yemen, Egypt, Algeria, Syria, Jordan, Morocco i Saudi Arabia), en canvi, pandas ordena els valors de forma alfabètica. 
Per tal de realitzar el meu gràfic, marcaré al dataframe de tots els països (ja excloent les regions) els 12 països amb major diferencia, per tal d'usar-los en el gràfic millorat.



In [18]:
#Marquem els 12 països seleccionats
df_mundial["Top12"] = df_mundial["Country Code"].isin(
    df_top12_mundial["Country Code"]
)

#Comprovem quins països han quedat seleccionats
df_mundial[df_mundial["Top12"]][["Country Name", "Country Code", "Difference"]].sort_values("Difference",ascending=False)


,Country Name,Country Code,Difference
0,Afghanistan,AFG,66.025
78,Iraq,IRQ,63.841
77,"Iran, Islamic Rep.",IRN,58.242
125,Oman,OMN,57.829
2,Algeria,DZA,57.544
180,"Yemen, Rep.",YEM,57.462
126,Pakistan,PAK,57.398
48,"Egypt, Arab Rep.",EGY,54.220
112,Morocco,MAR,52.300
159,Syrian Arab Republic,SYR,50.746


Tal i com s'observa, el país dels 12 seleccionats que té una diferència més baixa és Aràbia Saudita, amb 48,568 punts percentuals de diferència, tal i com es pot observar en el següent codi.
He decidit mantenir els 12 països que apareixen en el gràfic original per poder comparar-los, tot i que no considero que hi hagi un motiu concret per seleccionar-ne 12 i no, per exemple, 10. Ara bé, ja que se'n seleccionen 12, m'ha semblat interessant poder mostrar aquest valor en el gràfic que faré posteriorment. D'aquesta manera, l'usuari pot veure directament el llindar de diferència que s'ha utilitzat per seleccionar els 12 països.

In [19]:
llindar_top12 = df_top12_mundial["Difference"].min()

print(f"El llindar dels 12 països seleccionats és de {llindar_top12:.3f} punts percentuals.")

El llindar dels 12 països seleccionats és de 48.568 punts percentuals.


## GRÀFIC MILLORAT

In [20]:
# Creem el mapa mundial amb la diferència entre homes i dones
fig = px.choropleth(
    df_mundial,
    locations="Country Code",
    color="Difference",
    hover_name="Country Name",
    hover_data={
        "Female": True,
        "Male": True,
        "Difference": True,
        "Country Code": False
    },
    color_continuous_scale="YlOrRd",
    labels={
        "Difference": "Diferència (punts percentuals)",
        "Female": "Participació femenina (%)",
        "Male": "Participació masculina (%)"
    },
    title="El que les barres no ensenyen: diferències en la participació laboral"
)

fig.show()

Al principi vaig plantejar-me pintar tots els països segons la diferència entre la participació laboral masculina i femenina. Tot i això, vaig decidir no fer-ho perquè crec que hi hauria massa informació visual alhora i seria més difícil identificar els 12 països que estic analitzant.  
Per això, he decidit deixar la resta del mapa en un color neutre i aplicar una gradació de color només als 12 països amb una diferència més gran dins del conjunt de dades analitzat. Així, puc veure d’un cop d’ull que les diferències entre aquests 12 països no són exactament iguals i, al mateix temps, destacar la seva ubicació geogràfica.
Aquesta decisió també està relacionada amb els principis de la Gestalt, especialment amb la similitud i l’agrupament. Els 12 països comparteixen una mateixa codificació visual mitjançant el color, mentre que la resta del mapa queda en segon pla. Això fa que visualment es percebin com el grup que estem analitzant i facilita que l’usuari centri l’atenció en ells.


In [21]:
fig = px.choropleth(
    df_mundial,
    # Utilitzem el codi de cada país per situar-lo correctament al mapa
    locations="Country Code", #indiquem a quin país correspon cada fila
    color="Difference", #com donem color al mapa:s segons la diferència d'homes i dones
    hover_name="Country Name", #indiquem que es mostra al passar el ratolí per sobre
    title="Gender Gap in Labour Force Participation (2023)",
    labels={"Difference": "Gender gap (percentage points)"}
)


fig.update_layout(
    font=dict(family="Times New Roman",size=14),
    title=dict(font=dict(size=22),
               x=0.5,              #Com que el mapa està en el centre, centre també el títol
               xanchor="center"),   #El centre del títol és el punt de referència)

)

fig.show()

Separo els 12 països que volem destacar de la resta perquè després pugui donar-los un tractament visual diferent: els 12 tindran color segons la diferència i la resta quedarà en gris com a context.
Com que a partir d'ara treballo en dues capes, s'usa import plotly.graph_objects as go, que permet realitzar gràfics més personalitzats,  enlloc de import plotly.express as px, que són més directes.
Uso Choropleth, que és simplement el tipus de gràfic de Plotly que ens permet pintar països d'un mapa segons unes dades.


In [80]:

import plotly.graph_objects as go
#Obtenció dels dataframes de les dues capes
df_mapa_top12 = df_mundial[df_mundial["Top12"]].copy() #Els 12 països a destacar
df_mapa_resta = df_mundial[~df_mundial["Top12"]].copy() #La resta de països

#Creem un mapa buit on després afegirem les capes
fig = go.Figure()
fig.add_trace(                         #afegir una nova capa al mapa
    go.Choropleth(                     
        locations=df_mapa_resta["Country Code"],  #Països a pintar (la resta)
        z=df_mapa_resta["Difference"],            #Guardar la diferència real de cada país
        colorscale=[ #Tots els altres països queden de gris perquè l'inici i el final de l'escala són el mateix
            [0, "lightgray"],  #0 = inici de l'escala 
            [1, "lightgray"]   #1 = final de l'escala
        ],  
        showscale=False                #No cal mostrar una escala de colors per aquesta capa
    )
)
#Afegir els 12 països que volem destacar
fig.add_trace(
    go.Choropleth(
        locations=df_mapa_top12["Country Code"],  #Indicar quins són els països
        z=df_mapa_top12["Difference"],            #El color dependrà de la seva diferència
        colorscale=[ #Faig servir una escala de lila suau a lila intens
            [0, "plum"],    
            [1, "indigo"]        
        ],
        colorbar=dict(
            title="Gender gap (pp)",
            x=1.05,  #Moc la llegenda una mica cap a l'esquerra perquè quedava desquadrat
            len=0.5  #Faig la llegenda més curta respecte a l'alçada del gràfic
        )                   
    )
)

fig.add_annotation(
    text="12 countries highlighted with a difference ≥ 48.6 pp",  #Text que volem mostrar --> centrat (x) aprop de la part inferior (y)
    x=0.5,
    y=0.08,
    xref="paper",           #0 i 1 fan referència a l'amplada de tota la figura
    yref="paper",           #0 i 1 fan referència a l'alçada de tota la figura
    showarrow=False,        #No necessitem cap fletxa
    font=dict(
        family="Times New Roman",
        size=14
    )
)
#Afegeixo el meu tìtol
fig.update_layout(
    width=1100,   #Amplada del gràfic
    height=800,    #Alçada del gràfic,
    title=dict(
        text="Top 12 Countries with the Largest Gender Gap in Labour Force Participation"
              "<br><sup>Labour force participation rate, ages 15-64, 2023. Difference between male and female rates in percentage points.</sup>",
        x=0.5,  # Centrem el títol
        y=0.90,              # Baixem una mica el bloc de text
        yanchor="top",
        xanchor="center",
        font=dict(
            family="Times New Roman",
            size=30
        )
    )
)

fig.show()

A l’hora d’escollir el títol em vaig trobar amb el dubte de si podia mantenir el títol original: "TOP 12 COUNTRIES WHERE MEN OUTNUMBER WOMEN IN THE WORKFORCE". Al principi em semblava una bona opció perquè descrivia els països que volia destacar, però després, mirant les dades que estava utilitzant, vaig veure que podia donar a entendre una comparació de quantitats de persones, mentre que el nou mapa mostra la diferència entre dues taxes. Per això vaig decidir adaptar-lo perquè expliqués millor què estic representant jo.
Per exemple, a Afghanistan, la taxa de participació laboral és del 71,384% en els homes i del 5,359% en les dones, cosa que dona una diferència de 66,025 punts percentuals. Per tant, el mapa no està mostrant el nombre d’homes i dones que formen part de la força laboral, sinó la diferència entre les seves taxes de participació. 

Tot gris i només pintat els 12 països amb gradació. Quan fas hover sobre ell surt informació. 
També es poden seleccionar 2 països i es pot fer gràfic comparatiu --> més d'una vista --> nova pestanya i tirés endarrere 
Més vistes --> botó per canviar de vista
